In [2]:
!nvidia-smi

import sys
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("PyTorch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("Visible GPUs:", torch.cuda.device_count())
    print("GPU 0:", torch.cuda.get_device_name(0))

    free, total = torch.cuda.mem_get_info(0)
    print(f"GPU 0 free memory: {free / 1024**3:.2f} GiB")
    print(f"GPU 0 total memory: {total / 1024**3:.2f} GiB")

Thu Nov 26 20:43:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P0             27W /   70W |     105MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "Qwen/Qwen2.5-1.5B-Instruct"
device = "cuda:0"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float16,
    attn_implementation="sdpa",
).to(device).eval()

print("Transformers:", transformers.__version__)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Transformers: 5.20.2


In [4]:
print("Device:", model.device)
print("Weight dtype:", next(model.parameters()).dtype)
print(f"Allocated GPU memory: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GiB")

Device: cuda:0
Weight dtype: torch.float16
Allocated GPU memory: 2.88 GiB


In [5]:
messages = [
    {"role": "user", "content": "Explain a KV cache in two sentences."}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
inputs = tokenizer(text, return_tensors="pt").to(device)

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=64,
        do_sample=False,
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id,
    )

new_tokens = outputs[0, inputs["input_ids"].shape[1]:]
print(tokenizer.decode(new_tokens, skip_special_tokens=True))

A Key-Value (KV) cache is a type of data storage that stores key-value pairs for quick retrieval and updates. It allows efficient access to frequently used data without the need for complex queries or joins.


In [6]:
cfg = model.config

layers = cfg.num_hidden_layers
kv_heads = cfg.num_key_value_heads
head_dim = (
    getattr(cfg, "head_dim", None)
    or cfg.hidden_size // cfg.num_attention_heads
)
bytes_per_element = 2  # FP16 KV

# Each layer stores both a key and a value for every cached token.
kv_bytes_per_token = (
    2 * layers * kv_heads * head_dim * bytes_per_element
)

print("Layers:", layers)
print("KV heads:", kv_heads)
print("Head dimension:", head_dim)
print(f"KV per cached token: {kv_bytes_per_token / 1024:.1f} KiB")

for cached_tokens in [1024, 4096, 8192]:
    memory_mib = cached_tokens * kv_bytes_per_token / 1024**2
    print(f"{cached_tokens} cached tokens: {memory_mib:.1f} MiB")

Layers: 28
KV heads: 2
Head dimension: 128
KV per cached token: 28.0 KiB
1024 cached tokens: 28.0 MiB
4096 cached tokens: 112.0 MiB
8192 cached tokens: 224.0 MiB


In [7]:
import gc
import itertools
import json
import platform
import random
import statistics
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import torch
import transformers


# ---------- Configuration ----------
BATCH_SIZES = [1, 2, 4, 8]
LENGTHS = [256, 512, 1024, 2048]

WARMUP_RUNS = 10
MEASURED_RUNS = 50
GRID_SEED = 42

# Set only if you know the exact commit used to load the model.
MODEL_REVISION = None

DEVICE = torch.device("cuda:0")


# ---------- Validate the existing model ----------
assert torch.cuda.is_available(), "Enable the Kaggle GPU first."
assert not getattr(model, "is_quantized", False), "This sweep is labeled FP16."
assert all(p.device == DEVICE for p in model.parameters()), "Use GPU 0 only."
assert all(
    p.dtype == torch.float16
    for p in model.parameters()
    if p.is_floating_point()
), "Load the model in FP16 first."

assert WARMUP_RUNS >= 0 and MEASURED_RUNS >= 2
assert BATCH_SIZES and LENGTHS
assert all(
    isinstance(x, int) and x > 0
    for x in BATCH_SIZES + LENGTHS
)
assert max(LENGTHS) + 1 <= model.config.max_position_embeddings

model.eval()


# ---------- Create a separate folder for this execution ----------
run_id = datetime.now(timezone.utc).strftime(
    "fp16_%Y%m%dT%H%M%S_%fZ"
)
out_dir = Path("/kaggle/working/reserve_profiles") / run_id
out_dir.mkdir(parents=True, exist_ok=False)

token_id = tokenizer.encode(
    "hello", add_special_tokens=False
)[0]

cases = list(itertools.product(BATCH_SIZES, LENGTHS))

# Reproducible shuffled order reduces dependence on testing sizes
# in a consistently increasing order.
random.Random(GRID_SEED).shuffle(cases)


# ---------- Save environment and experiment settings ----------
metadata = {
    "run_id": run_id,
    "model_id": "Qwen/Qwen2.5-1.5B-Instruct",
    "model_revision": MODEL_REVISION,
    "revision_note": (
        "null means the existing model's exact commit was not recorded"
    ),
    "mode": "fp16",
    "python": platform.python_version(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "cuda_runtime": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(DEVICE),
    "gpu_total_bytes": (
        torch.cuda.get_device_properties(DEVICE).total_memory
    ),
    "warmup_runs": WARMUP_RUNS,
    "measured_runs": MEASURED_RUNS,
    "grid_seed": GRID_SEED,
    "case_order": cases,
    "input_token_id": token_id,
    "timing": (
        "synchronized forward-pass wall time including host launch overhead"
    ),
    "decode_setup": (
        "fresh prefill cache and argmax prepared outside the decode timer"
    ),
    "memory": (
        "per-trial PyTorch peaks; decode includes its pre-existing cache"
    ),
    "coverage": (
        "homogeneous batches; separate prefill/decode; "
        "no mixed-batch or switching costs"
    ),
    "model_config": model.config.to_dict(),
}

(out_dir / "metadata.json").write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8",
)


# ---------- Measure one independent trial ----------
@torch.inference_mode()
def measure_once(phase, ids, prompt_mask, decode_mask):
    prefix = cache = next_token = result = None

    try:
        if phase == "decode":
            # Rebuild the original-length cache for EVERY trial.
            # This work is outside the decode timer.
            prefix = model(
                input_ids=ids,
                attention_mask=prompt_mask,
                use_cache=True,
                logits_to_keep=1,
            )

            cache = prefix.past_key_values
            assert cache is not None

            next_token = prefix.logits[:, -1, :].argmax(
                dim=-1, keepdim=True
            )
            assert tuple(next_token.shape) == (ids.shape[0], 1)

            prefix = None

        torch.cuda.synchronize(DEVICE)
        torch.cuda.reset_peak_memory_stats(DEVICE)

        start = time.perf_counter()

        if phase == "prefill":
            result = model(
                input_ids=ids,
                attention_mask=prompt_mask,
                use_cache=True,
                logits_to_keep=1,
            )
        else:
            result = model(
                input_ids=next_token,
                attention_mask=decode_mask,
                past_key_values=cache,
                use_cache=True,
                logits_to_keep=1,
            )

        torch.cuda.synchronize(DEVICE)
        elapsed_ms = (time.perf_counter() - start) * 1000

        assert result.logits.shape[0] == ids.shape[0]

        return {
            "latency_ms": elapsed_ms,
            "peak_allocated_mib": (
                torch.cuda.max_memory_allocated(DEVICE) / 1024**2
            ),
            "peak_reserved_mib": (
                torch.cuda.max_memory_reserved(DEVICE) / 1024**2
            ),
        }

    finally:
        prefix = cache = next_token = result = None


# ---------- Summarize measurements for one configuration ----------
def summarize(samples):
    if not samples:
        return {"measured_runs": 0}

    values = [r["latency_ms"] for r in samples]

    if len(values) > 1:
        q1, _, q3 = statistics.quantiles(
            values, n=4, method="inclusive"
        )
        std = statistics.stdev(values)
    else:
        q1 = q3 = values[0]
        std = None

    mean = statistics.mean(values)

    return {
        "measured_runs": len(values),
        "median_ms": statistics.median(values),
        "mean_ms": mean,
        "std_ms": std,
        "q1_ms": q1,
        "q3_ms": q3,
        "iqr_ms": q3 - q1,
        "min_ms": min(values),
        "max_ms": max(values),
        "cv_pct": (
            100 * std / mean
            if std is not None and mean
            else None
        ),
        "peak_allocated_mib": max(
            r["peak_allocated_mib"] for r in samples
        ),
        "peak_reserved_mib": max(
            r["peak_reserved_mib"] for r in samples
        ),
    }


# ---------- Run the complete grid ----------
summary_rows = []
raw_rows = []

raw_columns = [
    "run_id", "mode", "phase", "batch_size", "length_tokens",
    "trial", "latency_ms", "peak_allocated_mib", "peak_reserved_mib",
]

print(f"Saving to: {out_dir}", flush=True)
print(
    f"{len(cases) * 2} cases; "
    f"{MEASURED_RUNS} measured trials per case",
    flush=True,
)

for batch_size, length in cases:
    for phase in ("prefill", "decode"):
        case = {
            "run_id": run_id,
            "mode": "fp16",
            "phase": phase,
            "batch_size": batch_size,
            "length_tokens": length,
        }

        # Fresh lists prevent mixing samples between configurations.
        samples = []
        status = "ok"
        error = ""
        warmups_completed = 0
        ids = prompt_mask = decode_mask = None

        try:
            ids = torch.full(
                (batch_size, length),
                token_id,
                dtype=torch.long,
                device=DEVICE,
            )
            prompt_mask = torch.ones_like(ids)
            decode_mask = torch.ones(
                (batch_size, length + 1),
                dtype=torch.long,
                device=DEVICE,
            )

            for _ in range(WARMUP_RUNS):
                measure_once(phase, ids, prompt_mask, decode_mask)
                warmups_completed += 1

            for trial in range(MEASURED_RUNS):
                measurement = measure_once(
                    phase, ids, prompt_mask, decode_mask
                )
                samples.append(measurement)

                raw_rows.append({
                    **case,
                    "trial": trial + 1,
                    **measurement,
                })

        except torch.cuda.OutOfMemoryError as exc:
            status = "oom"
            error = str(exc).splitlines()[0]

        finally:
            ids = prompt_mask = decode_mask = None
            gc.collect()
            torch.cuda.empty_cache()

        row = {
            **case,
            "status": status,
            "error": error,
            "warmups_completed": warmups_completed,
            **summarize(samples),
        }
        summary_rows.append(row)

        # Save completed work after every phase/configuration.
        # File writing happens outside the measured GPU calls.
        pd.DataFrame(summary_rows).to_csv(
            out_dir / "summary.csv", index=False
        )
        pd.DataFrame(raw_rows, columns=raw_columns).to_csv(
            out_dir / "raw_timings.csv", index=False
        )

        if samples:
            print(
                f"{phase:7} | B={batch_size:2} L={length:4} | "
                f"{status:3} | n={len(samples):2} | "
                f"median={row['median_ms']:.2f} ms | "
                f"IQR={row['iqr_ms']:.2f} ms",
                flush=True,
            )
        else:
            print(
                f"{phase:7} | B={batch_size:2} L={length:4} | "
                f"{status} | {error}",
                flush=True,
            )


# ---------- Display and save the final summary ----------
summary_df = pd.DataFrame(summary_rows).sort_values(
    ["phase", "batch_size", "length_tokens"]
)

summary_df.to_csv(out_dir / "summary.csv", index=False)

display(summary_df.round(3))
print(f"Files saved in: {out_dir}")

Saving to: /kaggle/working/reserve_profiles/fp16_20261126T204405_157396Z
32 cases; 50 measured trials per case
prefill | B= 2 L=2048 | ok  | n=50 | median=1354.04 ms | IQR=78.79 ms
decode  | B= 2 L=2048 | ok  | n=50 | median=43.32 ms | IQR=0.36 ms
prefill | B= 4 L= 512 | ok  | n=50 | median=404.76 ms | IQR=2.79 ms
decode  | B= 4 L= 512 | ok  | n=50 | median=36.65 ms | IQR=1.02 ms
prefill | B= 2 L= 512 | ok  | n=50 | median=203.69 ms | IQR=1.25 ms
decode  | B= 2 L= 512 | ok  | n=50 | median=35.89 ms | IQR=1.00 ms
prefill | B= 2 L=1024 | ok  | n=50 | median=505.12 ms | IQR=2.55 ms
decode  | B= 2 L=1024 | ok  | n=50 | median=36.46 ms | IQR=0.92 ms
prefill | B= 8 L=1024 | ok  | n=50 | median=1961.97 ms | IQR=10.74 ms
decode  | B= 8 L=1024 | ok  | n=50 | median=64.60 ms | IQR=0.27 ms
prefill | B= 4 L=1024 | ok  | n=50 | median=975.62 ms | IQR=1.84 ms
decode  | B= 4 L=1024 | ok  | n=50 | median=43.36 ms | IQR=0.44 ms
prefill | B= 8 L= 256 | ok  | n=50 | median=354.23 ms | IQR=1.73 ms
decode 

,run_id,mode,phase,batch_size,length_tokens,status,error,warmups_completed,measured_runs,median_ms,mean_ms,std_ms,q1_ms,q3_ms,iqr_ms,min_ms,max_ms,cv_pct,peak_allocated_mib,peak_reserved_mib
29,fp16_20261126T204405_157396Z,fp16,decode,1,256,ok,,10,50,35.142,35.334,0.829,34.734,35.710,0.976,34.177,37.383,2.347,2967.228,3182.0
17,fp16_20261126T204405_157396Z,fp16,decode,1,512,ok,,10,50,35.329,35.628,1.428,34.919,35.765,0.846,34.193,43.361,4.009,2978.513,3210.0
19,fp16_20261126T204405_157396Z,fp16,decode,1,1024,ok,,10,50,35.737,35.991,1.252,35.152,36.149,0.997,34.622,41.465,3.479,3004.046,3352.0
31,fp16_20261126T204405_157396Z,fp16,decode,1,2048,ok,,10,50,36.290,36.444,0.656,35.990,36.858,0.869,35.409,37.723,1.801,3092.591,3754.0
25,fp16_20261126T204405_157396Z,fp16,decode,2,256,ok,,10,50,35.711,36.354,2.794,35.035,36.666,1.632,34.546,53.060,7.684,2978.583,3186.0
5,fp16_20261126T204405_157396Z,fp16,decode,2,512,ok,,10,50,35.890,36.462,2.868,35.526,36.530,1.003,34.929,55.580,7.865,3004.114,3328.0
7,fp16_20261126T204405_157396Z,fp16,decode,2,1024,ok,,10,50,36.459,36.904,2.450,36.052,36.975,0.923,35.273,52.586,6.638,3092.634,3534.0
1,fp16_20261126T204405_157396Z,fp16,decode,2,2048,ok,,10,50,43.316,43.270,0.267,43.063,43.425,0.362,42.709,43.753,0.617,3146.829,4352.0
15,fp16_20261126T204405_157396Z,fp16,decode,4,256,ok,,10,50,36.047,36.195,0.684,35.760,36.589,0.829,35.115,37.869,1.889,3004.251,3268.0
3,fp16_20261126T204405_157396Z,fp16,decode,4,512,ok,,10,50,36.649,36.735,0.892,36.078,37.094,1.016,35.576,39.544,2.429,3092.720,3450.0


Files saved in: /kaggle/working/reserve_profiles/fp16_20261126T204405_157396Z
